# Bibliography Extraction Workflow

##input:
`.txt` files written by Isabella's workflow

In [ ]:
from pathlib import Path
import re
import pandas as pd

TXT_ROOT = Path("youthnex_txt_spaces")
EXPORT_PREFIX = "biblio_stabilized"

## 1. Load files

In [ ]:
def load_txt_corpus(txt_root: Path) -> pd.DataFrame:
    txt_root = Path(txt_root)

    if not txt_root.exists():
        raise FileNotFoundError(
            f"TXT_ROOT does not exist: {txt_root.resolve()}\n"
            "Point TXT_ROOT to the folder containing the preprocessed .txt files."
        )

    rows = []
    for txt_file in sorted(txt_root.rglob("*.txt")):
        rel = txt_file.relative_to(txt_root)
        text = txt_file.read_text(encoding="utf-8", errors="ignore")
        year_guess = txt_file.parent.name

        rows.append({
            "doc_id": txt_file.stem,
            "doc_uid": str(rel.with_suffix("")),
            "year": year_guess,
            "path": str(txt_file),
            "text": text,
        })

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No .txt files found under: {txt_root.resolve()}")

    return df

df_corpus = load_txt_corpus(TXT_ROOT)

print("Rows loaded:", len(df_corpus))
print("Unique doc_uid:", df_corpus["doc_uid"].nunique())
print("Unique doc_id:", df_corpus["doc_id"].nunique())
df_corpus.head()

## 2. Normalize text

- `text_multiline` to keep section/header structure
- `text_flat` is for fallback regex checks

In [ ]:
def normalize_text_multiline(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [ln.rstrip() for ln in text.split("\n")]

    cleaned = []
    blank_count = 0
    for ln in lines:
        if ln.strip() == "":
            blank_count += 1
            if blank_count <= 2:
                cleaned.append("")
        else:
            blank_count = 0
            cleaned.append(ln)

    return "\n".join(cleaned).strip()


def normalize_text_flat(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


df_corpus["text_multiline"] = df_corpus["text"].fillna("").apply(normalize_text_multiline)
df_corpus["text_flat"] = df_corpus["text"].fillna("").apply(normalize_text_flat)

df_corpus[["doc_uid", "year", "path"]].head()

## 3. Quick file diagnostics

In [ ]:
PUB_ANYWHERE_RE = re.compile(
    r"publicat|bibliograph|journal article|peer[- ]reviewed|refereed|book chapters?",
    flags=re.IGNORECASE
)

diag_rows = []
for _, row in df_corpus.iterrows():
    text = row["text_multiline"]
    m = PUB_ANYWHERE_RE.search(text)
    diag_rows.append({
        "doc_uid": row["doc_uid"],
        "has_pub_keyword_anywhere": bool(m),
        "n_lines": len(text.split("\n")) if isinstance(text, str) and text else 0,
        "n_blank_lines": sum(1 for ln in text.split("\n") if not ln.strip()) if isinstance(text, str) and text else 0,
    })

df_diag = pd.DataFrame(diag_rows)
print("Documents with publication-like keyword anywhere:", int(df_diag["has_pub_keyword_anywhere"].sum()))
print("Median lines per doc:", int(df_diag["n_lines"].median()))
print("Median blank lines per doc:", int(df_diag["n_blank_lines"].median()))
df_diag.head()

In [ ]:
def preview_document(doc_uid: str, max_chars: int = 4000):
    row = df_corpus.loc[df_corpus["doc_uid"] == doc_uid].iloc[0]
    print(row["text_multiline"][:max_chars])

# Set to a doc_uid value to inspect a specific file
example_doc_uid = None

if example_doc_uid:
    preview_document(example_doc_uid)
else:
    print("Set example_doc_uid to inspect a specific document.")

## 4. Strict publication header detection

- the extractor only starts on a **header-like line**
- noisy phrases like `scholarship` or `works in progress` are excluded from first-pass starts

In [ ]:
STRONG_PUBLICATION_HEADER_PATTERNS = [
    r"publications?",
    r"selected publications?",
    r"selected peer[- ]reviewed publications?",
    r"peer[- ]reviewed publications?",
    r"refereed publications?",
    r"journal articles?",
    r"peer[- ]reviewed journal articles?",
    r"refereed journal articles?",
    r"refereed articles?",
    r"articles in refereed journals?",
    r"book chapters?",
    r"books?\s+and\s+book\s+chapters?",
    r"bibliography",
    r"selected works",
    r"selected scholarly publications?",
    r"peer[- ]reviewed articles?(?:\s+and\s+chapters?)?",
    r"articles?,?\s+chapters?,?\s+and\s+reviews?",
    r"publications?\s+and\s+presentations?",
    r"published manuscripts?",
    r"academic publications?",
    r"scholarly works?",
    r"scholarly activity",
    r"scholarship",
    r"works? published",
    r"creative works?",
    r"monographs?",
    r"edited volumes?",
    r"reports?\s+and\s+publications?",
]

OTHER_SECTION_PATTERNS = [
    r"education",
    r"employment",
    r"academic appointments?",
    r"professional appointments?",
    r"professional experience",
    r"research",
    r"research support",
    r"sponsored research",
    r"grants?",
    r"awards?",
    r"honors?",
    r"teaching",
    r"service",
    r"professional service",
    r"editorial",
    r"editorial service",
    r"presentations?",
    r"conference presentations?",
    r"invited talks?",
    r"talks?",
    r"professional affiliations?",
    r"licensure",
    r"community engagement",
    r"media coverage",
    r"research funding",
    r"funding",
    # NOTE: standalone r"books?" removed — too broad, catches "Book" sub-headers inside pub sections
    r"edited books?",
    r"book reviews?",
    r"encyclopedia entries?",
    r"working papers?",
    r"manuscripts? (?:under review|in progress|in preparation|in prep)",
    r"works? in progress",
    r"research interests?",
    r"research experience",
    r"grants? and contracts?",
    r"fellowships?",
    r"professional activities",
    r"leadership",
    r"mentoring",
    r"training",
    r"media",
    r"outreach",
    r"extension",
    r"consulting"
]

PUB_HEADER_PREFIX_RE = re.compile(
    r"^(?:selected\s+)?(?:peer[- ]reviewed\s+|refereed\s+)?"
    r"(?:publications?|journal articles?|refereed journal articles?|peer[- ]reviewed journal articles?|"
    r"refereed articles?|articles in refereed journals?|book chapters?|books?\s+and\s+book\s+chapters?|"
    r"bibliography|scholarly works?|scholarly activity|scholarship|works? published|creative works?|"
    r"monographs?|edited volumes?|reports?\s+and\s+publications?|selected works|"
    r"selected scholarly publications?|peer[- ]reviewed articles?(?:\s+and\s+chapters?)?|"
    r"articles?,?\s+chapters?,?\s+and\s+reviews?|publications?\s+and\s+presentations?|"
    r"published manuscripts?|academic publications?)"
    r"(?:\s*[:\-\u2013\u2014\u2020\u2021*†].*)?$",
    flags=re.IGNORECASE,
)

OTHER_HEADER_PREFIX_RE = re.compile(
    r"^(?:" + "|".join(OTHER_SECTION_PATTERNS) + r")(?:\s*[:\-\u2013\u2014\u2020\u2021*†].*)?$",
    flags=re.IGNORECASE,
)

YEAR_TOKEN_RE = re.compile(
    r"\b(?:19|20)\d{2}\b|\bin press\b|\bforthcoming\b",
    flags=re.IGNORECASE,
)


def normalize_line(line: str) -> str:
    s = str(line).strip()
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_header_candidate(line: str) -> str:
    s = normalize_line(line)
    s = re.sub(r"^[\-\u2022\*\d\.\)\(\s]+", "", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def line_is_header_like(line: str) -> bool:
    s = normalize_header_candidate(line)
    if not s:
        return False
    if len(s) > 160:
        return False
    if s.endswith("."):
        return False
    if len(s.split()) > 20:
        return False
    alpha_chars = sum(ch.isalpha() for ch in s)
    if alpha_chars < 4:
        return False
    all_caps = s.upper() == s and alpha_chars >= 4
    titleish = bool(re.fullmatch(r"[A-Z][A-Za-z0-9/&,()'\-\u2013\u2014:* ]+", s))
    return all_caps or titleish


def classify_line(line: str):
    s = normalize_header_candidate(line)
    if not s:
        return None
    if PUB_HEADER_PREFIX_RE.match(s):
        return "publication"
    if OTHER_HEADER_PREFIX_RE.match(s) and line_is_header_like(s):
        return "other"
    return None


In [ ]:
def inspect_candidate_headers(text_multiline: str, max_lines: int = 80) -> pd.DataFrame:
    rows = []
    for i, line in enumerate(text_multiline.split("\n")[:max_lines]):
        raw = line.strip()
        if not raw:
            continue

        rows.append({
            "line_idx": i,
            "raw_line": raw,
            "header_class": classify_line(raw),
            "header_like": line_is_header_like(raw),
        })

    return pd.DataFrame(rows)

for _, row in df_corpus.head(3).iterrows():
    print("\nDOC:", row["doc_uid"])
    display(inspect_candidate_headers(row["text_multiline"], max_lines=80))

## 5. Publication section extraction

The extractor finds the first strong publication header line, slices until the next strong non-publication header, and avoids starting too early on a non-header line

In [ ]:
def extract_all_publication_sections(text_multiline: str):
    '''
    Extract ALL publication sections from a CV, not just the first.
    Returns list of {section_label, section_start_line, section_end_line, section_text}.
    Handles CVs with sub-sections like Peer-Reviewed Articles, Book Chapters, etc.
    '''
    if not isinstance(text_multiline, str) or not text_multiline.strip():
        return []
    lines = text_multiline.split('\n')
    sections = []
    i = 0
    while i < len(lines):
        cls = classify_line(lines[i])
        if cls == 'publication':
            start_line = i
            section_label = normalize_line(lines[i])
            end_line = len(lines)
            for j in range(start_line + 1, len(lines)):
                if classify_line(lines[j]) == 'other':
                    end_line = j
                    break
            section_lines = lines[start_line:end_line]
            section_text = '\n'.join(section_lines).strip()
            nonempty = [ln for ln in section_lines if ln.strip()]
            if len(nonempty) >= 2:
                sections.append({
                    'section_label': section_label,
                    'section_start_line': start_line,
                    'section_end_line': end_line,
                    'section_text': section_text,
                })
            i = end_line
        else:
            i += 1
    return sections


In [ ]:
for _, row in df_corpus.head(5).iterrows():
    sections = extract_all_publication_sections(row['text_multiline'])
    print('DOC:', row['doc_uid'])
    if sections:
        for sec in sections:
            print('SECTION LABEL:', sec['section_label'])
            print('LINE RANGE:', sec['section_start_line'], 'to', sec['section_end_line'])
            print(sec['section_text'][:1400])
            print()
    else:
        print('No publication section found')
    print('\n' + '=' * 100 + '\n')


## 6. Citation splitting

In [ ]:
YEAR_RE = r"(?:19|20)\d{2}|in press|forthcoming"
NUMBER_OR_BULLET_RE = re.compile(r"^(?:\[?\d+\]?\.?\s+|[-\u2022*]\s+)")
LEADING_AUTHOR_YEAR_RE = re.compile(
    rf"^(?:[A-Z][A-Za-z'`\-]+(?:,?\s+[A-Z]\.)?(?:,\s*[A-Z]\.)*.+?\b(?:{YEAR_RE})\b)",
    flags=re.IGNORECASE,
)

_HEADER_STRIP_RE = re.compile(
    r"^(?:selected\s+)?(?:peer[- ]reviewed\s+|refereed\s+)?"
    r"(?:publications?|journal articles?|refereed journal articles?|peer[- ]reviewed journal articles?|"
    r"refereed articles?|articles in refereed journals?|book chapters?|books?\s+and\s+book\s+chapters?|"
    r"bibliography|scholarly works?|scholarly activity|scholarship|works? published|creative works?|"
    r"monographs?|edited volumes?|reports?\s+and\s+publications?)"
    r"\s*[:\-\u2013\u2014\u2020\u2021*†]?\s*",
    flags=re.IGNORECASE,
)


def clean_citation_text(text: str) -> str:
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"^[-\u2022*\s]+", "", text)
    return text


def remove_header_from_section(section_text: str):
    lines = [ln.rstrip() for ln in section_text.split("\n")]
    if not lines:
        return []

    # Strip ALL consecutive pub sub-headers from the top (not just the first).
    # Handles cases like "SCHOLARSHIP\nPEER REVIEWED ARTICLES:\nActual citation..."
    while lines:
        first = normalize_header_candidate(lines[0])
        if classify_line(first) == "publication":
            stripped = _HEADER_STRIP_RE.sub("", lines[0], count=1)
            if stripped.strip():
                lines[0] = stripped
                break  # remainder of line is content, stop stripping
            else:
                lines = lines[1:]  # entire line was a header, remove and continue
        else:
            break

    return [ln.rstrip() for ln in lines]


def _split_block_by_boundaries(block_lines):
    # Split a blank-line block further on number/bullet/author-year boundaries.
    groups = []
    current = []
    for line in block_lines:
        stripped = line.strip()
        if not stripped:
            continue
        starts_new = (
            current
            and (NUMBER_OR_BULLET_RE.match(stripped) or LEADING_AUTHOR_YEAR_RE.match(stripped))
        )
        if starts_new:
            groups.append(current)
            current = [NUMBER_OR_BULLET_RE.sub("", stripped).strip()]
        else:
            if current:
                current.append(stripped)
            else:
                current.append(NUMBER_OR_BULLET_RE.sub("", stripped).strip())
    if current:
        groups.append(current)
    return groups


def split_candidate_citations(section_text: str):
    if not isinstance(section_text, str) or not section_text.strip():
        return []
    content_lines = remove_header_from_section(section_text)
    if not content_lines:
        return []

    blocks = []
    current_block = []
    for line in content_lines:
        if not line.strip():
            if current_block:
                blocks.append(current_block)
                current_block = []
        else:
            current_block.append(line.strip())
    if current_block:
        blocks.append(current_block)

    if len(blocks) > 1:
        entries = []
        for block in blocks:
            for grp in _split_block_by_boundaries(block):
                entries.append(clean_citation_text(" ".join(grp)))
        return [e for e in entries if len(e) >= 25]

    entries = []
    current = []
    for line in content_lines:
        stripped = line.strip()
        if not stripped:
            continue
        starts_new = (
            not current
            or NUMBER_OR_BULLET_RE.match(stripped)
            or LEADING_AUTHOR_YEAR_RE.match(stripped)
        )
        cleaned = NUMBER_OR_BULLET_RE.sub("", stripped).strip()
        if starts_new:
            if current:
                entries.append(clean_citation_text(" ".join(current)))
            current = [cleaned]
        else:
            current.append(cleaned)
    if current:
        entries.append(clean_citation_text(" ".join(current)))
    return [e for e in entries if len(e) >= 25]


In [ ]:
for _, row in df_corpus.head(5).iterrows():
    sections = extract_all_publication_sections(row['text_multiline'])
    print('DOC:', row['doc_uid'])
    if sections:
        for sec in sections:
            citations = split_candidate_citations(sec['section_text'])
            print('SECTION LABEL:', sec['section_label'])
            print('N CANDIDATES:', len(citations))
            for i, citation in enumerate(citations[:3], start=1):
                print(f'\nCitation {i}:')
                print(citation[:500])
            print()
    else:
        print('No publication section found')
    print('\n' + '=' * 100 + '\n')


## 7. Full extraction across the corpus

In [ ]:
results = []

for _, row in df_corpus.iterrows():
    text_multi = row['text_multiline']
    sections = extract_all_publication_sections(text_multi)
    has_pub_keyword_anywhere = bool(PUB_ANYWHERE_RE.search(text_multi))

    if not sections:
        results.append({
            'doc_uid': row['doc_uid'],
            'doc_id': row['doc_id'],
            'year': row['year'],
            'path': row['path'],
            'has_pub_keyword_anywhere': has_pub_keyword_anywhere,
            'section_found': False,
            'section_label': None,
            'section_start_line': None,
            'section_end_line': None,
            'citation_id': None,
            'raw_citation': None,
            'n_candidate_citations': 0,
        })
        continue

    for section_result in sections:
        citations = split_candidate_citations(section_result['section_text'])
        if not citations:
            results.append({
                'doc_uid': row['doc_uid'],
                'doc_id': row['doc_id'],
                'year': row['year'],
                'path': row['path'],
                'has_pub_keyword_anywhere': has_pub_keyword_anywhere,
                'section_found': True,
                'section_label': section_result['section_label'],
                'section_start_line': section_result['section_start_line'],
                'section_end_line': section_result['section_end_line'],
                'citation_id': None,
                'raw_citation': None,
                'n_candidate_citations': 0,
            })
            continue
        for i, citation in enumerate(citations, start=1):
            results.append({
                'doc_uid': row['doc_uid'],
                'doc_id': row['doc_id'],
                'year': row['year'],
                'path': row['path'],
                'has_pub_keyword_anywhere': has_pub_keyword_anywhere,
                'section_found': True,
                'section_label': section_result['section_label'],
                'section_start_line': section_result['section_start_line'],
                'section_end_line': section_result['section_end_line'],
                'citation_id': i,
                'raw_citation': citation,
                'n_candidate_citations': len(citations),
            })

df_biblio_candidates = pd.DataFrame(results)
df_biblio_candidates.head(10)


## 8. Quality flags and sanity checks

In [ ]:
def citation_quality_flag(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return "missing"

    lowered = text.lower()
    has_year = bool(re.search(r"\b(?:19|20)\d{2}\b|\bin press\b|\bforthcoming\b", lowered))
    too_short = len(text) < 40
    too_long = len(text) > 1200

    if too_short:
        return "too_short"
    if too_long:
        return "too_long"
    if not has_year:
        return "no_year_found"
    return "ok"


df_biblio_candidates["citation_quality_flag"] = (
    df_biblio_candidates["raw_citation"].apply(citation_quality_flag)
)

print("Files loaded:", len(df_corpus))
print("Unique documents (doc_uid):", df_corpus["doc_uid"].nunique())
print(
    "Documents with publication-like keyword anywhere:",
    df_biblio_candidates.loc[df_biblio_candidates["has_pub_keyword_anywhere"], "doc_uid"].nunique()
)
print(
    "Documents with section found:",
    df_biblio_candidates.loc[df_biblio_candidates["section_found"], "doc_uid"].nunique()
)
print(
    "Total candidate citation rows:",
    df_biblio_candidates["raw_citation"].notna().sum()
)

print("\nTop detected section labels")
display(
    df_biblio_candidates
    .groupby("section_label", dropna=False)
    .size()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("n_rows")
)

print("\nCitation quality flags")
display(
    df_biblio_candidates
    .groupby("citation_quality_flag", dropna=False)
    .size()
    .sort_values(ascending=False)
    .to_frame("n_rows")
)

### Sanity-check sample

The top `section_label` values should look like real headers such as:
- `PUBLICATIONS`
- `Selected Publications`
- `Journal Articles`

If you see names, addresses, or long prose, the start detector is still too loose.

In [ ]:
sample_sections = (
    df_biblio_candidates.loc[df_biblio_candidates["section_found"], ["doc_uid", "section_label"]]
    .drop_duplicates()
    .head(25)
)
sample_sections

## 9. Review tables

In [ ]:
needs_review_no_section = (
    df_biblio_candidates[
        (df_biblio_candidates["has_pub_keyword_anywhere"]) &
        (~df_biblio_candidates["section_found"])
    ][["doc_uid", "year", "path"]]
    .drop_duplicates()
)

needs_review_no_citations = (
    df_biblio_candidates[
        (df_biblio_candidates["section_found"]) &
        (df_biblio_candidates["n_candidate_citations"] == 0)
    ][["doc_uid", "year", "path", "section_label"]]
    .drop_duplicates()
)

problem_citations = (
    df_biblio_candidates[
        df_biblio_candidates["citation_quality_flag"].isin(["too_short", "too_long", "no_year_found"])
    ][["doc_uid", "citation_id", "citation_quality_flag", "raw_citation"]]
)

print("Needs review - pub keyword but no section:", len(needs_review_no_section))
print("Needs review - section but no citations:", len(needs_review_no_citations))
print("Problem citations:", len(problem_citations))

In [ ]:
needs_review_no_section.head(20)

In [ ]:
needs_review_no_citations.head(20)

In [ ]:
problem_citations.head(20)

## 10. Export

In [ ]:
candidate_path   = f"{EXPORT_PREFIX}_candidate_citations.csv"
no_section_path  = f"{EXPORT_PREFIX}_needs_review_no_section.csv"
no_citations_path= f"{EXPORT_PREFIX}_needs_review_no_citations.csv"
problem_path     = f"{EXPORT_PREFIX}_problem_citations.csv"

# Exclude rows with no citation text before export
df_export = df_biblio_candidates[
    df_biblio_candidates["citation_quality_flag"] != "missing"
].copy()

df_export.to_csv(candidate_path, index=False)
needs_review_no_section.to_csv(no_section_path, index=False)
needs_review_no_citations.to_csv(no_citations_path, index=False)
problem_citations.to_csv(problem_path, index=False)

print("Saved:")
print("-", candidate_path, f"({len(df_export):,} rows, {len(df_biblio_candidates) - len(df_export)} missing rows excluded)")
print("-", no_section_path)
print("-", no_citations_path)
print("-", problem_path)
